# Agent Analytics Dashboard




#### Data Schema & Table Format
This dashboard is designed to visualize agent execution logs exported to BigQuery. The dashboard logic assumes your table follows the standard Agent Development Kit (ADK) Schema.

*Please refer to the official documentation:
👉 [ADK BigQuery Schema Reference](https://adk.dev/integrations/bigquery-agent-analytics/#schema-reference)*

Before running the cells below, ensure you have your Google Cloud details ready.

#### 🔑 Setup Instructions

1.   Locate Configuration: Scroll down to the Sidebar Configuration section within the "Webapp script" cell.
2.   Update Credentials: Replace the placeholder values for project_id, dataset_id, and table_id with your actual BigQuery details:

*   Project ID: Your GCP Project ID (e.g., my-project-123).
*   Dataset ID: The BigQuery dataset name.
*   Table ID: The specific table name containing your logs.

3. Execute: Once updated, run all cells in order.


##### Installing dependencies

In [ ]:
!pip install -q streamlit plotly pandas google-cloud-bigquery bigframes db-dtypes

##### Authentication

In [ ]:
from google.colab import auth
auth.authenticate_user()

##### Webapp script

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
from google.cloud import bigquery
import plotly.express as px
import re

# --- 1. Page Configuration ---
st.set_page_config(page_title="Agent Analytics V2", layout="wide")
st.title("🤖 Agent Analytics & Orchestration Dashboard")

# --- 2. Shared Query Fragment ---
ENRICHED_EVENTS_CTE = """
    SELECT
        *,
        CASE
            WHEN event_type = 'USER_MESSAGE_RECEIVED' THEN 'user'
            WHEN event_type IN ('INVOCATION_STARTING','INVOCATION_COMPLETED') THEN 'invocation'
            WHEN event_type IN ('AGENT_STARTING','AGENT_COMPLETED') THEN 'agent'
            WHEN event_type IN ('LLM_REQUEST','LLM_RESPONSE','LLM_ERROR') THEN 'llm'
            WHEN event_type IN ('TOOL_STARTING','TOOL_COMPLETED','TOOL_ERROR') THEN 'tool'
            WHEN event_type = 'STATE_DELTA' THEN 'state'
            WHEN event_type LIKE 'HITL_%' THEN 'hitl'
            ELSE 'other'
        END AS event_family,
        (status = 'ERROR' OR error_message IS NOT NULL OR event_type LIKE '%_ERROR') AS is_canonical_error
    FROM {table_ref}
    WHERE timestamp >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL @time_hours HOUR)
"""

# --- 3. Sidebar Configuration ---
with st.sidebar:
    st.header("Data Source")
    project_id = st.text_input("GCP Project ID", "REPLACE WITH YOUR PROJECT ID")
    dataset_id = st.text_input("Dataset ID", "REPLACE WITH YOUR DATASET ID")
    table_id = st.text_input("Table ID", "REPLACE WITH YOUR TABLE ID")

    # Identifier Validation
    for val in [project_id, dataset_id, table_id]:
        if not re.match(r'^[\w.-]+$', val):
            st.error(f"Invalid identifier: {val}")
            st.stop()

    table_ref = f"`{project_id}.{dataset_id}.{table_id}`"

    st.divider()
    st.header("Global Filters")
    time_hours = st.slider("Time Range (Hours)", 1, 720, 72)
    agent_name = st.text_input("Filter by Agent Name", "")

# --- 4. Data Connector ---
@st.cache_resource
def get_bq_client(p_id):
    return bigquery.Client(project=p_id)

client = get_bq_client(project_id)

def safe_query(sql, params=None):
    try:
        job_config = bigquery.QueryJobConfig(query_parameters=params) if params else None
        formatted_sql = sql.replace("{table_ref}", table_ref)
        return client.query(formatted_sql, job_config=job_config).to_dataframe()
    except Exception as e:
        st.error(f"Query Failed: {e}")
        return pd.DataFrame()

# --- 5. Global Params ---
global_params = [bigquery.ScalarQueryParameter("time_hours", "INT64", time_hours)]
where_clause = "timestamp >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL @time_hours HOUR)"
if agent_name:
    where_clause += " AND agent = @agent_name"
    global_params.append(bigquery.ScalarQueryParameter("agent_name", "STRING", agent_name))

# --- 6. Executive Summary ---
st.header("📊 Executive Summary")
kpi_data = safe_query(f"""
    SELECT
        COUNT(DISTINCT session_id) as sessions,
        COUNT(DISTINCT trace_id) as traces,
        COUNTIF(event_type = 'LLM_REQUEST') as llm_calls,
        SAFE_DIVIDE(COUNTIF(status = 'ERROR' OR error_message IS NOT NULL), COUNT(*)) as error_rate
    FROM {{table_ref}} WHERE {where_clause}
""", params=global_params)

if not kpi_data.empty:
    c1, c2, c3, c4 = st.columns(4)
    c1.metric("Sessions", kpi_data['sessions'][0])
    c2.metric("Traces", kpi_data['traces'][0])
    c3.metric("LLM Calls", kpi_data['llm_calls'][0])
    c4.metric("Error Rate", f"{kpi_data['error_rate'][0]*100:.2f}%")

st.divider()

# --- 7. Detailed Tabs ---
tabs = st.tabs(["📈 Usage", "⚡ Performance", "🛡️ Reliability", "🖼️ Multimodal", "🤝 HITL", "🔍 Trace Explorer"])

# TAB 0: USAGE & VOLUME
with tabs[0]:
    st.subheader("Token Consumption")
    token_trend = safe_query(f"""
        SELECT TIMESTAMP_TRUNC(timestamp, HOUR) as hour,
               SUM(CAST(JSON_VALUE(content, '$.usage.prompt') AS INT64)) as prompt,
               SUM(CAST(JSON_VALUE(content, '$.usage.completion') AS INT64)) as completion
        FROM {{table_ref}} WHERE event_type = 'LLM_RESPONSE' AND {where_clause}
        GROUP BY 1 ORDER BY 1
    """, params=global_params)
    if not token_trend.empty:
        fig = px.area(token_trend, x="hour", y=["prompt", "completion"], title="Tokens over Time")
        st.plotly_chart(fig, use_container_width=True)
    else:
        st.info("No token usage data found.")

# TAB 1: PERFORMANCE
with tabs[1]:
    st.subheader("Latency Metrics")
    lat_data = safe_query(f"""
        WITH enriched AS ({ENRICHED_EVENTS_CTE})
        SELECT
            APPROX_QUANTILES(CAST(JSON_VALUE(latency_ms, '$.total_ms') AS FLOAT64), 100)[OFFSET(50)] as p50,
            APPROX_QUANTILES(CAST(JSON_VALUE(latency_ms, '$.total_ms') AS FLOAT64), 100)[OFFSET(90)] as p90,
            AVG(CAST(JSON_VALUE(latency_ms, '$.time_to_first_token_ms') AS FLOAT64)) as avg_ttft
        FROM enriched WHERE event_family = 'llm'
        {" AND agent = @agent_name" if agent_name else ""}
    """, params=global_params)
    if not lat_data.empty and pd.notnull(lat_data['p50'][0]):
        p1, p2, p3 = st.columns(3)
        p1.metric("p50 Latency", f"{lat_data['p50'][0]:.0f}ms")
        p2.metric("p90 Latency", f"{lat_data['p90'][0]:.0f}ms")
        p3.metric("Avg TTFT", f"{lat_data['avg_ttft'][0]:.0f}ms")
    else:
        st.warning("No performance data found.")

# TAB 2: RELIABILITY
with tabs[2]:
    st.subheader("Error Distribution")
    err_data = safe_query(f"""
        WITH enriched AS ({ENRICHED_EVENTS_CTE})
        SELECT event_family, event_type, COUNT(*) as errors
        FROM enriched WHERE is_canonical_error = TRUE
        {" AND agent = @agent_name" if agent_name else ""}
        GROUP BY 1, 2 ORDER BY errors DESC
    """, params=global_params)
    if not err_data.empty:
        fig = px.bar(err_data, x="event_type", y="errors", color="event_family")
        st.plotly_chart(fig, use_container_width=True)
    else:
        st.success("No errors detected.")

# TAB 3: MULTIMODAL
with tabs[3]:
    st.subheader("Media Parts")
    multi_data = safe_query(f"""
        SELECT cp.mime_type, COUNT(*) as count
        FROM {{table_ref}}, UNNEST(content_parts) as cp
        WHERE {where_clause} GROUP BY 1
    """, params=global_params)
    if not multi_data.empty:
        fig = px.pie(multi_data, values='count', names='mime_type')
        st.plotly_chart(fig, use_container_width=True)
    else:
        st.info("No multimodal content (images/audio) found.")

# TAB 4: HITL
with tabs[4]:
    st.subheader("Human-in-the-Loop")
    hitl_data = safe_query(f"""
        WITH enriched AS ({ENRICHED_EVENTS_CTE})
        SELECT event_type, COUNT(*) as count
        FROM enriched WHERE event_family = 'hitl'
        {" AND agent = @agent_name" if agent_name else ""}
        GROUP BY 1
    """, params=global_params)
    if not hitl_data.empty:
        st.table(hitl_data)
    else:
        st.info("No HITL events found.")

# TAB 5: TRACE EXPLORER
with tabs[5]:
    st.subheader("Session Explorer")
    sessions = safe_query(f"""
        SELECT session_id, agent, MIN(timestamp) as ts,
        FORMAT_TIMESTAMP('%m-%d %H:%M', MIN(timestamp)) as f_ts
        FROM {{table_ref}} WHERE {where_clause} GROUP BY 1, 2 ORDER BY ts DESC LIMIT 50
    """, params=global_params)

    if not sessions.empty:
        sessions['label'] = sessions['f_ts'] + " | " + sessions['agent'].fillna("?") + " | " + sessions['session_id'].str[:8]
        label_map = dict(zip(sessions['label'], sessions['session_id']))
        sel_label = st.selectbox("Select Session", sessions['label'])
        sel_id = label_map[sel_label]

        trace_detail = safe_query("""
            SELECT timestamp, event_type, agent, status, error_message
            FROM {table_ref} WHERE session_id = @sid ORDER BY timestamp
        """, params=[bigquery.ScalarQueryParameter("sid", "STRING", sel_id)])
        st.dataframe(trace_detail, use_container_width=True)

##### Run app.py

In [ ]:
import subprocess, re, time
from IPython.display import clear_output

# Cloudflare Installation
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# Start Streamlit (logs hidden in streamlit_logs.txt)
with open("streamlit_logs.txt", "w") as f:
    subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501",
                      "--server.enableCORS=false", "--server.enableXsrfProtection=false"],
                     stdout=f, stderr=f)

# Start Tunnel
p = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8501"],
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

for line in p.stdout:
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        clear_output()
        print(f"✅ Dashboard Live: \033[1;34m{match.group(0)}\033[0m")
        break